#### Importing required libraries 

In [1]:
import pandas as pd
from sklearn.metrics import *
from tqdm import tqdm
from utils import Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

#### Testing a single load 

In [2]:
train_dataset = 'charlie_hebdo'
test_dataset = 'ottawashooting'
time_cut =3*60*24
processor = Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning(train_dataset,\
           test_dataset, time_cut=time_cut,test_size=0.7)

processor.load_data()
processor.process_data()
train,test = processor.get_final_dataframes()

rumour
1    307
0    293
Name: count, dtype: int64


In [3]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_test =test['rumour']

#### Example  training

In [6]:
# Compute class weights to handle imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))


In [7]:
# Compute class weights to handle imbalance
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# Initialize Random Forest with class weights
model = RandomForestClassifier(
    n_estimators=250,
    max_depth=2,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight=class_weight_dict,  # Handles imbalance
    n_jobs=-1,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict and evaluate

y_proba_test = model.predict_proba(X_test)[:, 1]  # For ROC AUC

y_proba_train = model.predict_proba(X_train)[:, 1]  # For ROC AUC

In [8]:

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = [f1_score(y_train, (y_proba_train > t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_train_pred = (y_proba_train > best_threshold).astype(int)
y_test_pred = (y_proba_test > best_threshold).astype(int)

# Evaluation function
def evaluate(y_true, y_pred, y_prob, label=""):
    print(f"  - Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  - Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  - Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  - AUC:       {roc_auc_score(y_true, y_prob):.4f}")
    print("")

# Show metrics
evaluate(y_train, y_train_pred, y_proba_train, label="Train")
evaluate(y_test, y_test_pred, y_proba_test, label="Test")

  - Accuracy:  0.8420
  - Precision: 0.6749
  - Recall:    0.7755
  - AUC:       0.8969

  - Accuracy:  0.7017
  - Precision: 0.8636
  - Recall:    0.4951
  - AUC:       0.8337



#### Setting MLflow Experiment

In [2]:
mlflow.set_experiment("Random Forest  2025-11-05 Sydney Siege TF")

2025/11/16 12:39:58 INFO mlflow.tracking.fluent: Experiment with name 'Random Forest  2025-11-05 Sydney Siege TF' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/96', creation_time=1763296798035, experiment_id='96', last_update_time=1763296798035, lifecycle_stage='active', name='Random Forest  2025-11-05 Sydney Siege TF', tags={}>

#### Loading dataset statistics to get the final time cut 

In [8]:
df_posts_by_time_cut = pd.read_csv('sydneysiege_posts_by_time_cut.csv')

In [9]:
time_cut_last_post = int(df_posts_by_time_cut[df_posts_by_time_cut.post==\
                         int(df_posts_by_time_cut['post'].max())].time_cut.min())

In [10]:
time_cut_last_post

1016

* **The initial  time cut will be 10 minutes after the first post publication**
*  **The final time cut will be equal to 6 hours after the publication of last post**

In [11]:
previous_node_count = 0

for time_cut in range(15, time_cut_last_post+(60*6), 5):
    print(f"\n=== Time Cut: {time_cut} ===")
    
    train_dataset = 'charlie_hebdo'
    test_dataset = 'sydneysiege'
    #test_dataset = 'ottawashooting'
    #test_dataset = 'germanwings_crash'
    #test_dataset = 'ferguson'
    time_cut =time_cut
    processor = Load_Rumours_Dataset_filtering_since_first_post_Transfer_Learning(train_dataset,\
               test_dataset, time_cut=time_cut,test_size=0.7)
    
    processor.load_data()
    processor.process_data()
    train,test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    # Compute class weights to handle imbalance
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, class_weights))
    
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=2,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight=class_weight_dict,
        n_jobs=-1,
        random_state=42
    )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_train, y_train)


        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        if X_test_new.shape[0] >0:
            y_test_new_prob = model.predict_proba(X_test_new)[:, 1]

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 15 ===
rumour
0    8
1    5
Name: count, dtype: int64
New Instances: 13

=== Time Cut: 20 ===
rumour
0    13
1     6
Name: count, dtype: int64
New Instances: 6

=== Time Cut: 25 ===
rumour
0    15
1     7
Name: count, dtype: int64
New Instances: 3

=== Time Cut: 30 ===
rumour
0    20
1    12
Name: count, dtype: int64
New Instances: 10

=== Time Cut: 35 ===
rumour
0    24
1    15
Name: count, dtype: int64
New Instances: 7

=== Time Cut: 40 ===
rumour
0    26
1    16
Name: count, dtype: int64
New Instances: 3

=== Time Cut: 45 ===
rumour
0    33
1    17
Name: count, dtype: int64
New Instances: 8

=== Time Cut: 50 ===
rumour
0    39
1    19
Name: count, dtype: int64
New Instances: 8

=== Time Cut: 55 ===
rumour
0    39
1    21
Name: count, dtype: int64
New Instances: 2

=== Time Cut: 60 ===
rumour
0    42
1    21
Name: count, dtype: int64
New Instances: 3

=== Time Cut: 65 ===
rumour
0    44
1    23
Name: count, dtype: int64
New Instances: 4

=== Time Cut: 70 ===
rumour
0  

In [5]:
2

2